# SENTIMENT ANALYSIS — IndoBERT BASE (NON FINE-TUNED)
Frozen IndoBERT + trainable classifier head only.
Lower-bound baseline for comparison vs Fine-Tuned IndoBERT.

In [ ]:
!pip install -q -U "transformers<5.0.0" "accelerate>=0.26.0" "datasets" "packaging<25.0,>=23.2"

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from collections import Counter
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_recall_fscore_support
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)
from transformers.modeling_outputs import SequenceClassifierOutput

import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Imports complete. Running on: {device}')

In [ ]:
SEED = 42
MODEL_NAME = 'indobenchmark/indobert-base-p1'
MAX_LENGTH = 512

DATA_PATH = '/kaggle/input/datasets/davinraffilio9/datalabeled'
MODEL_PATH = '/kaggle/working'

set_seed(SEED)

print(f'Model: IndoBERT BASE (Frozen BERT + Trainable Classifier)')
print(f'Max Length: {MAX_LENGTH}')

In [ ]:
df1 = pd.read_csv(f'{DATA_PATH}/detik_labeled.csv')
df2 = pd.read_csv(f'{DATA_PATH}/cnbc_labeled.csv')
df3 = pd.read_csv(f'{DATA_PATH}/kompas_labeled.csv')

print(f'Detik: {len(df1)} | CNBC: {len(df2)} | Kompas: {len(df3)}')

required_cols = ['date', 'title', 'content', 'article_id', 'text', 'label']
df_seed = pd.concat(
    [df[required_cols].copy() for df in [df1, df2, df3]],
    ignore_index=True
)
print(f'Total Combined Samples: {len(df_seed)}')

In [ ]:
# Train/Val Split
train_df, val_df = train_test_split(
    df_seed, test_size=0.2,
    stratify=df_seed['label'], random_state=SEED
)

# Remap labels (-1, 0, 1) -> (0, 1, 2)
label_to_id = {-1: 0, 0: 1, 1: 2}
id_to_label = {0: 'negative', 1: 'neutral', 2: 'positive'}

train_df['label_id'] = train_df['label'].map(label_to_id)
val_df['label_id'] = val_df['label'].map(label_to_id)

# Class weights with amplification [1.5, 1.0, 1.5] (same as GRU/CNN/FT)
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_df['label_id']),
    y=train_df['label_id']
)
class_weights = class_weights * np.array([1.5, 1.0, 1.5])
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print(f'Train: {len(train_df)} | Val: {len(val_df)}')
print(f'Class weights (amplified): {class_weights}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text, truncation=True, max_length=self.max_length,
            padding=False, return_tensors=None
        )
        return {
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask'],
            'labels': label
        }

train_dataset = SentimentDataset(
    train_df['text'].tolist(), train_df['label_id'].tolist(),
    tokenizer, MAX_LENGTH
)
val_dataset = SentimentDataset(
    val_df['text'].tolist(), val_df['label_id'].tolist(),
    tokenizer, MAX_LENGTH
)
print('✅ Tokenizer and Dataset ready.')

In [ ]:
# IndoBERT Base — BERT FROZEN, only classifier trainable
class IndoBERTBase(nn.Module):
    """
    Frozen IndoBERT → [CLS] → Dropout → Linear(768, 3)
    Key difference vs Fine-Tuned: BERT weights are FROZEN.
    """
    def __init__(self, model_name, num_labels, dropout_rate=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        # FREEZE all BERT parameters
        for param in self.bert.parameters():
            param.requires_grad = False
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
    
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        with torch.no_grad():
            outputs = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
            loss = loss_fct(logits.view(-1, self.classifier.out_features), labels.view(-1))
        return SequenceClassifierOutput(
            loss=loss, logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

In [ ]:
# Focal Loss Trainer (sama dengan GRU/CNN/FT)
class FocalLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_gamma = focal_gamma
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        ce_loss = F.cross_entropy(logits, labels, weight=self.class_weights, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.focal_gamma * ce_loss).mean()
        return (focal_loss, outputs) if return_outputs else focal_loss

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    f1_per_class = f1_score(labels, preds, average=None)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
        'f1_negative': f1_per_class[0],
        'f1_neutral': f1_per_class[1],
        'f1_positive': f1_per_class[2],
    }

In [ ]:
# Initialize model with FROZEN BERT
sentiment_model = IndoBERTBase(
    model_name=MODEL_NAME,
    num_labels=3,
    dropout_rate=0.1
).to(device)

total_p = sum(p.numel() for p in sentiment_model.parameters())
train_p = sum(p.numel() for p in sentiment_model.parameters() if p.requires_grad)
print(f'Total params: {total_p:,}')
print(f'Trainable params (classifier only): {train_p:,} ({train_p/total_p*100:.4f}%)')

# Training Arguments — PERSIS SAMA dengan GRU/CNN/FT
training_args = TrainingArguments(
    output_dir=f'{MODEL_PATH}/sentiment_training',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    save_total_limit=1,
    learning_rate=3e-5,
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_dir=f'{MODEL_PATH}/logs',
    logging_steps=50,
    report_to='none',
    fp16=True,
    dataloader_num_workers=2,
    seed=SEED,
)

trainer = FocalLossTrainer(
    class_weights=class_weights_tensor,
    focal_gamma=2.0,
    model=sentiment_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
print('✅ Trainer Initialized.')

In [ ]:
print('=' * 40)
print('🚀 STARTING TRAINING')
print('=' * 40)
trainer.train()
print('✅ Training Complete.')

In [ ]:
print('Running Final Evaluation...')
pred = trainer.predict(val_dataset)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

final_metrics = trainer.evaluate()
print('\n' + '=' * 50)
print('FINAL RESULTS — IndoBERT Base (Frozen)')
print('=' * 50)
print(f"Accuracy    : {final_metrics['eval_accuracy']:.4f}")
print(f"Macro F1    : {final_metrics['eval_macro_f1']:.4f}")
print(f"Weighted F1 : {final_metrics['eval_weighted_f1']:.4f}")
print('-' * 50)
print(classification_report(y_true, y_pred, target_names=['negative','neutral','positive'], digits=4))

In [ ]:
# Save model (state_dict, NOT trainer.save_model() — custom nn.Module!)
import json
os.makedirs(f'{MODEL_PATH}/indobert_base_best', exist_ok=True)
torch.save(sentiment_model.state_dict(),
           f'{MODEL_PATH}/indobert_base_best/model_state_dict.pt')
tokenizer.save_pretrained(f'{MODEL_PATH}/indobert_base_best')
print(f'[OK] Model saved → {MODEL_PATH}/indobert_base_best/model_state_dict.pt')

summary = {
    'model': 'IndoBERT Base (Frozen / Non Fine-Tuned)',
    'architecture': 'Frozen IndoBERT → [CLS] → Dropout(0.1) → Linear(768,3)',
    'bert_frozen': True,
    'indobert': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'dropout': 0.1,
    'results': {
        'accuracy': float(final_metrics['eval_accuracy']),
        'macro_f1': float(final_metrics['eval_macro_f1']),
        'weighted_f1': float(final_metrics['eval_weighted_f1']),
    },
    'data': {'train_samples': len(train_df), 'val_samples': len(val_df)},
}
with open(f'{MODEL_PATH}/indobert_base_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'[OK] Summary saved → {MODEL_PATH}/indobert_base_summary.json')
print(json.dumps(summary['results'], indent=2))

In [ ]:
# Save predictions for cross-model comparison (McNemar test)
np.save(f'{MODEL_PATH}/base_val_preds.npy', np.array(y_pred))
np.save(f'{MODEL_PATH}/base_val_true.npy', np.array(y_true))
print(f'Saved: {MODEL_PATH}/base_val_preds.npy')
print(f'Saved: {MODEL_PATH}/base_val_true.npy')

# Ablation Study 1 — Dropout Rate (R10 - Reviewer Response)

In [ ]:
print('⏭️ Ablation 1 (Dropout Variants) — SKIPPED, re-enable later')

# Ablation Study 2 — Learning Rate (R10 - Reviewer Response)

In [ ]:
print('⏭️ Ablation 2 (Learning Rate) — SKIPPED, re-enable later')

# Ablation Study 3 — Loss Function CE vs Focal (R6 - Reviewer Response)

In [ ]:
print('⏭️ Ablation 3 (CE vs Focal Loss) — SKIPPED, re-enable later')

# Stratified 5-Fold Cross-Validation (R10 - Reviewer Response)

In [ ]:
print('⏭️ K-Fold CV — SKIPPED, re-enable later')

# Computational Efficiency Analysis (R9 - Reviewer Response)

In [ ]:
print('⏭️ Efficiency Analysis — SKIPPED, re-enable later')

# Statistical Significance Testing (R7 - Reviewer Response)

In [ ]:
print('⏭️ Statistical Significance — SKIPPED, re-enable later')